# MultiChainComparison

O `MultiChainComparison` permite comparar múltiplas tentativas de resolução de um mesmo problema.

O processo possui duas etapas:

1. `ChainOfThought` gera várias soluções independentes para o mesmo problema.
2. `MultiChainComparison` recebe essas soluções e realiza uma nova chamada ao LLM para:
   - comparar os raciocínios;
   - identificar inconsistências;
   - corrigir possíveis erros;
   - produzir uma resposta final.

Fluxo:

Pergunta  
   ↓  
ChainOfThought  
   ↓  
Tentativa 1  
Tentativa 2  
Tentativa 3  
Tentativa 4  
Tentativa 5  
   ↓  
MultiChainComparison  
   ↓  
Resposta final  

In [1]:
import os
from dotenv import load_dotenv
import dspy

load_dotenv()

True

## Setup - Configuração do Modelo

Carregamos as variáveis de ambiente do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [2]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Configura o modelo padrão utilizado pelo DSPy
dspy.configure(lm=lm)

## Definindo o problema

Vamos utilizar uma Signature semelhante à utilizada anteriormente com
`ChainOfThought`.

A diferença é que escolheremos um problema combinatório um pouco mais
complexo, no qual diferentes cadeias de raciocínio podem chegar a resultados
diferentes.

In [3]:
class CalculadoraMatematica(dspy.Signature):
    """Resolva o problema matemático apresentado."""

    # Pergunta que deverá ser resolvida
    pergunta: str = dspy.InputField(
        desc="Problema matemático a ser resolvido"
    )

    # Resultado final
    resposta: str = dspy.OutputField(
        desc="Resposta numérica final. Retorne apenas o número."
    )

In [4]:
pergunta = """
Quantos números inteiros de cinco algarismos distintos podem ser formados
utilizando os dígitos de 0 a 9, sabendo que o número deve:

1. ter exatamente cinco algarismos;
2. possuir todos os algarismos distintos;
3. ser maior que 30000;
4. ser divisível por 5?

Retorne apenas a quantidade final.
"""

## Gerando múltiplas soluções

Antes de utilizar o `MultiChainComparison`, precisamos gerar diferentes
tentativas de solução.

Usaremos `ChainOfThought` com `n=5`.

Isso faz com que o modelo produza cinco completions para a mesma pergunta.

A temperatura é aumentada para permitir alguma diversidade entre as soluções.
Essa diversidade é importante: se todas as cadeias forem idênticas, há pouco
a ser comparado.

In [5]:
# Número de raciocínios que serão produzidos
M = 5


# Campo de raciocínio personalizado.
#
# O MultiChainComparison atual utiliza apenas a primeira linha do reasoning
# de cada tentativa.
#
# Por isso pedimos explicitamente que o raciocínio seja escrito em um único
# parágrafo, sem quebras de linha.
campo_raciocinio = dspy.OutputField(
    desc=(
        "Explique detalhadamente o raciocínio matemático em um único "
        "parágrafo, sem utilizar quebras de linha."
    )
)


# Cria o ChainOfThought responsável por gerar as diferentes soluções
gerador = dspy.ChainOfThought(
    CalculadoraMatematica,
    rationale_field=campo_raciocinio,
)


# Gera M soluções diferentes.
#
# n=M solicita múltiplas completions.
# temperature=1.0 aumenta a diversidade entre as tentativas.
resultado_tentativas = gerador(
    pergunta=pergunta,
    config={
        "n": M,
        "temperature": 1.0,
    },
)


# Recupera todas as completions produzidas
tentativas = resultado_tentativas.completions

## Visualizando as tentativas

Antes de executar o MultiChainComparison, é interessante observar as soluções
produzidas pelo ChainOfThought.

Idealmente veremos pequenas diferenças de abordagem — e eventualmente
diferenças na resposta final.

In [6]:
for i, tentativa in enumerate(tentativas, start=1):

    print("=" * 80)
    print(f"TENTATIVA {i}")
    print("=" * 80)

    print("\nRaciocínio:")
    print(tentativa.reasoning)

    print("\nResposta:")
    print(tentativa.resposta)

    print()

TENTATIVA 1

Raciocínio:
Para ser divisível por 5 o último dígito deve ser 0 ou 5. Se o último dígito for 0, o primeiro (não nulo) pode ser 3,4,5,6,7,8,9 (7 opções); os três dígitos do meio devem ser distintos entre si e diferentes do primeiro e do zero, portanto P(8,3)=8·7·6=336 arranjos; total neste caso 7·336=2352. Se o último dígito for 5, o primeiro pode ser 3,4,6,7,8,9 (não pode ser 0 nem 5), 6 opções; os três dígitos do meio são escolhidos entre os restantes 8 dígitos distintos e ordenados, P(8,3)=336, dando 6·336=2016. Somando os dois casos obtemos 2352+2016=4368.

Resposta:
4368

TENTATIVA 2

Raciocínio:
Como o número deve ser divisível por 5, o último algarismo só pode ser 0 ou 5; além disso o primeiro algarismo (dezena de milhar) deve ser 3,4,5,6,7,8 ou 9 para ser maior que 30000. Caso 1: último algarismo 0 — o primeiro algarismo tem 7 possibilidades (3–9) e os três algarismos do meio devem ser distintos entre si e distintos do primeiro e do zero, ou seja, escolhidos e orden

## MultiChainComparison

Agora entregamos as cinco tentativas ao `MultiChainComparison`.

O parâmetro `M` deve obrigatoriamente ser igual ao número de tentativas
fornecido.

O módulo analisará as soluções anteriores e realizará uma nova inferência
para produzir uma resposta consolidada.

In [8]:
# Cria o módulo responsável por comparar as soluções
comparador = dspy.MultiChainComparison(
    CalculadoraMatematica,
    M=M,
    temperature=1.0,  # GPT-5 Mini aceita apenas temperature=1
)


# Compara as M soluções geradas anteriormente
resultado_final = comparador(
    completions=tentativas,
    pergunta=pergunta,
)

## Resultado final

Além da resposta definida originalmente na Signature, o
`MultiChainComparison` adiciona um campo `rationale`.

Esse campo contém o raciocínio utilizado para consolidar as diferentes
tentativas.

In [9]:
print("=" * 80)
print("RESULTADO DO MULTICHAINCOMPARISON")
print("=" * 80)

print("\nRaciocínio consolidado:")
print(resultado_final.rationale)

print("\nResposta final:")
print(resultado_final.resposta)

RESULTADO DO MULTICHAINCOMPARISON

Raciocínio consolidado:
Para ser divisível por 5, o último dígito é 0 ou 5. O primeiro dígito (dezena de milhar) deve ser 3,4,5,6,7,8 ou 9 (maior que 30000).

Caso 1: último dígito = 0.
- Primeiro dígito: 7 opções (3–9).
- Restam 8 dígitos disponíveis para preencher, em ordem, as três posições do meio: P(8,3)=8·7·6=336.
- Total neste caso: 7·336 = 2352.

Caso 2: último dígito = 5.
- Primeiro dígito não pode ser 5, logo opções = {3,4,6,7,8,9} = 6.
- Para as três posições do meio há novamente P(8,3)=336 escolhas.
- Total neste caso: 6·336 = 2016.

Somando os dois casos: 2352 + 2016 = 4368.

Resposta final:
4368


In [10]:
print("=" * 80)
print("COMPARAÇÃO")
print("=" * 80)

for i, tentativa in enumerate(tentativas, start=1):
    print(f"Tentativa {i}: {tentativa.resposta}")

print("-" * 80)

print(f"MultiChainComparison: {resultado_final.resposta}")

print("-" * 80)

# Resposta correta conhecida do problema
resposta_correta = "4368"

print(f"Resposta esperada:     {resposta_correta}")
print(
    "Resultado correto:    ",
    resultado_final.resposta.strip() == resposta_correta,
)

COMPARAÇÃO
Tentativa 1: 4368
Tentativa 2: 4368
Tentativa 3: 4368
Tentativa 4: 4368
Tentativa 5: 4368
--------------------------------------------------------------------------------
MultiChainComparison: 4368
--------------------------------------------------------------------------------
Resposta esperada:     4368
Resultado correto:     True


## Inspecionando o histórico

O MultiChainComparison realiza uma segunda etapa de inferência.

Podemos utilizar `inspect_history()` para observar como as diferentes
tentativas foram transformadas em entradas para o LLM comparador.

In [11]:
# Mostra as duas últimas chamadas principais:
#
# 1. geração das múltiplas cadeias
# 2. comparação das cadeias
dspy.inspect_history(n=2)





[2026-09-07T10:51:22.928884]

System message:

Your input fields are:
1. `pergunta` (str): Problema matemático a ser resolvido
Your output fields are:
1. `reasoning` (str): Explique detalhadamente o raciocínio matemático em um único parágrafo, sem utilizar quebras de linha.
2. `resposta` (str): Resposta numérica final. Retorne apenas o número.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## pergunta ## ]]
{pergunta}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "resposta": "{resposta}"
}
In adhering to this structure, your objective is: 
        Resolva o problema matemático apresentado.


User message:

[[ ## pergunta ## ]]

Quantos números inteiros de cinco algarismos distintos podem ser formados
utilizando os dígitos de 0 a 9, sabendo que o número deve:

1. ter exatamente cinco algarismos;
2. possuir todos os algarismos distintos;
3. s